In [1]:
# prompt: use OpenAI GPT4o to extract first columns' paper title. My prompt: Please extract paper title from this sentence.

!pip uninstall -y openai
!pip install --upgrade openai

import openai
print(openai.__version__)
import pandas as pd

# Assuming 'senior_author' DataFrame is loaded and contains a column named 'Paper Title'

# Set your OpenAI API key
openai.# Replace with your actual API key

Found existing installation: openai 1.65.2
Uninstalling openai-1.65.2:
  Successfully uninstalled openai-1.65.2
  Using cached openai-1.65.2-py3-none-any.whl (473 kB)
1.65.2


/afs/csail.mit.edu/u/y/yuexing/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/afs/csail.mit.edu/u/y/yuexing/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
from openai import OpenAI  # This will cause an ImportError
# import openai

#We define which model to use throughout
MODEL = 'gpt-4o'
MAX_TOKENS = 8000
WAIT_TIME = 0.8 # Wait time between each request. This depends on the rate limit of the model used: GPT-4 needs longer wait time than GPT-3.5.

client = openai.OpenAI(
    #Set the API key. See the how-to guide for further instructions
)

In [3]:
import openai
print(openai.__version__)

# Retrieve available models
models = openai.models.list()

# Display model IDs
for model in models.data:
    print(model.id)

1.65.2
gpt-4.5-preview
omni-moderation-2024-09-26
gpt-4.5-preview-2025-02-27
gpt-4o-mini-audio-preview-2024-12-17
dall-e-3
dall-e-2
gpt-4o-audio-preview-2024-10-01
gpt-4o-audio-preview
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-realtime-preview
gpt-4-turbo-preview
o1-mini-2024-09-12
o1-preview-2024-09-12
o1-mini
gpt-4-0125-preview
o1-preview
gpt-4o-mini-audio-preview
whisper-1
gpt-4-turbo
gpt-4o-realtime-preview-2024-10-01
gpt-4
babbage-002
chatgpt-4o-latest
tts-1-hd-1106
gpt-4o-audio-preview-2024-12-17
o1
o1-2024-12-17
o3-mini-2025-01-31
tts-1-hd
o3-mini
text-embedding-3-large
tts-1
tts-1-1106
gpt-4-turbo-2024-04-09
davinci-002
gpt-3.5-turbo-1106
gpt-3.5-turbo-instruct
gpt-4o-2024-11-20
gpt-3.5-turbo-instruct-0914
gpt-3.5-turbo-0125
gpt-4o-realtime-preview-2024-12-17
gpt-3.5-turbo
gpt-4o-realtime-preview
gpt-3.5-turbo-16k
text-embedding-3-small
gpt-4-1106-preview
text-embedding-ada-002
gpt-4-0613
gpt-4o-mini-2024-07-18
gpt-4o-2024-05-13
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4o
om

In [4]:
import openai

# Define function to generate synthetic cases
def generate_synthetic_case():
    prompt = """
    Based on the NCCN Prostate Cancer Early Detection Guidelines, generate a synthetic clinical case for a patient undergoing prostate cancer screening. 
    The case should include age, PSA level, digital rectal exam (DRE) findings, family history, and any relevant risk factors. 
    Then, create a multiple-choice question with five answer choices, where only one is correct. 
    Format the output as follows:

    **Case Description & Question:** 
    A [age]-year-old [risk factor] male presents for routine prostate cancer screening. His PSA level is [PSA level] ng/mL, and his DRE findings are [DRE findings]. He has a [family history] of prostate cancer. 
    What is the most appropriate next step in his management?

    **Answer Choices:**
    A) [Choice A]
    B) [Choice B]
    C) [Choice C]
    D) [Choice D]
    E) [Choice E]

    **Correct Answer:**
    [Correct Answer]
    """

    client = openai.OpenAI(
    #Set the API key. See the how-to guide for further instructions
)  # Create an OpenAI client
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": "You are a medical AI assistant trained in prostate cancer early detection based on NCCN guidelines."},
                  {"role": "user", "content": prompt}]
    )

    output = response.choices[0].message.content.split("\n\n")
    
    # Extracting the relevant parts
    case_description = output[0].replace("**Case Description & Question:** ", "").strip()
    answer_choices = output[1].replace("**Answer Choices:**", "").strip()
    correct_answer = output[2].replace("**Correct Answer:**", "").strip()

    return case_description, answer_choices, correct_answer

In [5]:
# Generate multiple cases
num_cases = 10  # Adjust as needed
data = [generate_synthetic_case() for _ in range(num_cases)]

# Convert to DataFrame
df = pd.DataFrame(data, columns=["Case Description & Question", "Answer Choices", "Correct Answer"])

# Save to CSV
df.to_csv("Synthetic_Testing.csv", index=False)

print("Synthetic dataset 'Synthetic_Testing.csv' has been generated successfully!")


Synthetic dataset 'Synthetic_Testing.csv' has been generated successfully!


## Add Reasoning for each step

Based on the question/prompt and correct answer (generated by GPT-4o), generate a reasoning tree-based process that provide "Confidence" and "Difficulty" (generated by GPT-o1-mini, Claude, etc).

In [14]:
# Function to generate Tree-of-Thought reasoning and prediction using GPT-o1
def generate_treethought_reasoning(case_question, answer_choices):
    prompt = f"""
    Based on the following medical case:

    **Case Description & Question:**
    {case_question}

    **Answer Choices:**
    {', '.join(answer_choices)}

    **Task:**
    Please provide a structured Tree-of-Thought (ToT) reasoning process to determine the correct answer according to NCCN guidelines.

    **Required Structure:**
    1. **Stepwise Decision Path**: Explain each logical step in choosing the correct answer.
    2. **NCCN Justification**: Reference NCCN guidelines and provide a link to the relevant guideline.
    3. **Final Decision**: Conclude with the correct answer based on the reasoning.

    Only select one answer from the provided choices.
    """

    try:
        
        client = openai.OpenAI(
    #Set the API key. See the how-to guide for further instructions
)
        response = client.chat.completions.create(
            model="o1",
            messages=[{"role": "system", "content": "You are an expert oncology AI assistant following NCCN guidelines for medical decision-making."},
                      {"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content  # Extract GPT-o1's output

    except Exception as e:
        return f"Error generating reasoning: {str(e)}"

# Apply GPT-4o to generate reasoning for each case
df["Tree-of-Thought Reasoning & Predicted Answer"] = df.apply(lambda row: generate_treethought_reasoning(row["Case Description & Question"],
                                                                                                          row["Answer Choices"]), axis=1)


In [15]:
# Display full first 3 rows without truncation
pd.set_option('display.max_colwidth', None)  # Ensures full text display
print(df.head(3))  # Show first 3 rows

                                                                                                                                                                                                                                                                   Case Description & Question  \
0             A 55-year-old African American male presents for routine prostate cancer screening. His PSA level is 4.8 ng/mL, and his DRE findings are normal. He has a positive family history of prostate cancer in his father.  \nWhat is the most appropriate next step in his management?   
1  A 55-year-old African American male presents for routine prostate cancer screening. His PSA level is 4.5 ng/mL, and his DRE findings are normal. He has a family history of prostate cancer, with his father diagnosed at age 60. What is the most appropriate next step in his management?   
2       A 55-year-old African American male presents for routine prostate cancer screening. His PSA level is 2.8 ng/mL, and his DR

In [16]:
# Save results to CSV (Optional)
df.to_csv("nccn_treethought_reasoning_predictions.csv", index=False)


In [26]:
import json

# Function to generate step-wise reasoning using OpenAI o1-mini
def generate_reasoning_with_openai(case_description, correct_answer):
    prompt = f"""
    Given the following case description and correct answer:

    Case Description:
    {case_description}

    Correct Answer:
    {correct_answer}

    Provide a structured step-wise reasoning process in JSON format. Each step should contain:
    - "Step": A short description of the clinical decision-making process.
    - "Next Step": The decision taken at that step.
    - "Difficulty": Label as "Easy", "Medium", or "Hard".
    - "Confidence": Label as "Very Confident", "Medium Confident", or "No Confident".

    JSON Format Example:
    [
        {{
            "Step": "Initial evaluation of PSA levels and risk factors.",
            "Next Step": "Proceed to baseline PSA testing.",
            "Difficulty": "Easy",
            "Confidence": "Very Confident"
        }},
        {{
            "Step": "PSA result interpretation.",
            "Next Step": "PSA >3 ng/mL, so further evaluation required.",
            "Difficulty": "Medium",
            "Confidence": "Medium Confident"
        }},
        {{
            "Step": "Decision to perform biopsy.",
            "Next Step": "PSA >10 ng/mL, biopsy recommended.",
            "Difficulty": "Hard",
            "Confidence": "Very Confident"
        }}
    ]

    Provide only the JSON output.
    """

    client = openai.OpenAI(
    #Set the API key. See the how-to guide for further instructions
)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a medical AI assistant trained in prostate cancer early detection based on NCCN guidelines."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content  # Returns JSON-formatted reasoning

# Function to generate synthetic cases
def generate_synthetic_case():
    prompt = """
    Using the NCCN Prostate Cancer Early Detection Guidelines, generate a synthetic clinical case for a patient undergoing prostate cancer screening.
    The case should include age, PSA level, digital rectal exam (DRE) findings, family history, and relevant risk factors.
    Then, create a multiple-choice question with five answer choices, where only one is correct.
    """

    client = openai.OpenAI(
    #Set the API key. See the how-to guide for further instructions
)  
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a medical AI assistant trained in prostate cancer early detection based on NCCN guidelines."},
            {"role": "user", "content": prompt}
        ]
    )

    output = response.choices[0].message.content.split("\n\n")
    
    case_description = output[0].replace("**Case Description & Question:** ", "").strip()
    answer_choices = output[1].replace("**Answer Choices:**", "").strip()
    correct_answer = output[2].replace("**Correct Answer:**", "").strip()
    
    reasoning_steps = generate_reasoning(case_description, correct_answer)
    
    return case_description, answer_choices, correct_answer, reasoning_steps